# Static UMAP Plotting with Cross-Matched Overlays & Clustering Metrics

Extends the static UMAP overlay notebook with quantitative clustering metrics for evaluating how well cross-matched samples (mergers, lens candidates, LSBs, etc.) cluster in the latent space.

---

## Metric 1: MNLN Ratio (Median Nearest Labeled Neighbor)

For each labeled point (e.g., merger), find the distance to its closest *other* labeled point. Take the **median** of these distances. Divide by the expected median if labels were randomly assigned (via permutation test).

- **Ratio < 1** — labeled points are closer to each other than random. 0.5 means they are twice as close as expected by chance.
- **Ratio = 1** — no spatial preference, same as random.
- **Ratio > 1** — labeled points are more spread out than random (unlikely in practice).
- **p-value** — fraction of random permutations that produced an equally small or smaller median distance. Small p = the closeness is real, not luck.

The **median** makes this robust to the island problem: if 10 of 13 mergers cluster tightly on one island and 3 are stranded on another, the median reflects the tight group, not the cross-island outliers.

## Metric 2: CMC-Gini (Cluster Membership Concentration)

HDBSCAN partitions all UMAP points into clusters. Then we ask: how unevenly are the labeled points distributed across those clusters?

- **Gini = 0** — labeled points spread perfectly uniformly across all clusters (no preference).
- **Gini = 1** — all labeled points in a single cluster (maximum concentration).
- **Intermediate values** — e.g., 0.6 means labeled points are moderately concentrated in a few clusters.
- **p-value** — how often random label assignments produce a Gini at least this high. Small p = the concentration is real.
- **nClus** — how many HDBSCAN clusters were found. If this is very low (1-2), the Gini becomes uninformative.

## Reading them together

| MNLN | Gini | Interpretation |
|---|---|---|
| Low ratio, significant p | High, significant p | Strong clustering — labeled points are near each other AND concentrated in specific regions |
| Low ratio, significant p | Low / non-significant | Labeled points are locally close but don't prefer specific clusters (spread across several regions, but always near another labeled point) |
| High ratio / non-significant | High, significant p | Labeled points aren't especially close to each other, but they do land in the same few clusters (loosely occupying the same broad regions) |
| High ratio / non-significant | Low / non-significant | No clustering signal |

## 2D vs High-Dim comparison

Both metrics are computed in 2D UMAP space and (optionally) in the high-dimensional latent space.

- MNLN low in both — real proximity in the latent space, preserved by UMAP
- MNLN low in 2D only — UMAP projection artifact
- Gini high in HD but not 2D — latent space clusters mergers together but UMAP scrambles the cluster boundaries

---

**WARNING: Running HDBSCAN on high-dimensional latent vectors is significantly slower than on 2D UMAP coordinates. Expect the `include_highdim=True` panels to take considerably longer, especially for large datasets. If speed is a concern, run with `include_highdim=False` first to check 2D metrics, then selectively enable high-dim.**

In [ ]:
from __future__ import annotations

import logging
import re
from pathlib import Path
from typing import Iterable

import hyrax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import LogNorm, Normalize

from research_paths import paths
import static_umap_plotting as sup


In [ ]:
PROJECT_ROOT = Path.cwd()
HYRAX_RUN_BASE = paths.hyrax_runs
DEFAULT_N_PERMUTATIONS = 500
INCLUDE_HIGHDIM = False

# None means use every valid non-negative time-since-merger value.
# Set to a float such as 3.0 if you want to restrict the time-since overlays.
TIME_SINCE_MERGER_MAX_GYR = None


def first_existing_path(candidates: Iterable[str | Path]) -> Path:
    """Return the first existing path, or the first candidate for readable errors."""
    resolved = [Path(candidate).expanduser() for candidate in candidates]
    for path in resolved:
        if path.exists():
            return path
    return resolved[0]


def profile_path(name: str, fallback: str | Path) -> Path:
    """Read a path from research_paths.py, with a fallback for older profile files."""
    return Path(paths.as_dict().get(name, fallback)).expanduser()


SAMPLE_CATALOG_PATHS = {
    'all': paths.catalog('all'),
    'raw_merger_flags': paths.catalog('raw_merger_flags'),
    'le_120x120': paths.catalog('le_120x120'),
    'gt_120x120': paths.catalog('gt_120x120'),
}

XCATALOG_PATHS = {
    'rubin_euclid_vissyn': first_existing_path([
        profile_path('xmatch_catalog_rubin_euclid_vissyn', PROJECT_ROOT / 'data' / 'xmatched_cats' / 'rubin_euclid_vissyn.parquet'),
        '/mmfs1/gscratch/escience/aritrag/comcam_dp1/external_catalogs/xmatched_cats/rubin_euclid_vissyn.parquet',
    ]),
    'gzoo_decals': first_existing_path([
        profile_path('xmatch_catalog_gzoo_decals', PROJECT_ROOT / 'data' / 'xmatched_cats' / 'rubin_gzoo_decals_spatial.parquet'),
        '/mmfs1/gscratch/escience/aritrag/comcam_dp1/external_catalogs/xmatched_cats/rubin_gzoo_decals_spatial.parquet',
    ]),
    'euclid_lens': first_existing_path([
        profile_path('xmatch_catalog_euclid_lens', PROJECT_ROOT / 'data' / 'xmatched_cats' / 'rubin_euclid_lens_spatial.parquet'),
        '/mmfs1/gscratch/escience/aritrag/comcam_dp1/external_catalogs/xmatched_cats/rubin_euclid_lens_spatial.parquet',
    ]),
    'des_lsb': first_existing_path([
        profile_path('xmatch_catalog_des_lsb', PROJECT_ROOT / 'data' / 'xmatched_cats' / 'rubin_des_lsb.parquet'),
        '/mmfs1/gscratch/escience/aritrag/comcam_dp1/external_catalogs/xmatched_cats/rubin_des_lsb.parquet',
    ]),
}


def catalog_path_status() -> pd.DataFrame:
    """Show profile-managed sample catalogs and optional external xmatch catalogs."""
    rows = []
    for name, path in SAMPLE_CATALOG_PATHS.items():
        rows.append({'source': 'sample', 'name': name, 'path': str(path), 'exists': Path(path).exists()})
    for name, path in XCATALOG_PATHS.items():
        rows.append({'source': 'xmatch_optional', 'name': name, 'path': str(path), 'exists': Path(path).exists()})
    return pd.DataFrame(rows)


def load_external_catalog(catalog_path: str | Path) -> pd.DataFrame:
    """Load an overlay catalog from parquet, FITS, or CSV."""
    catalog_path = Path(catalog_path).expanduser()
    if not catalog_path.exists():
        raise FileNotFoundError(f'Catalog not found: {catalog_path}')

    suffix = catalog_path.suffix.lower()
    if suffix in {'.parquet', '.pq'}:
        return pd.read_parquet(catalog_path)
    if suffix in {'.fits', '.fit', '.fts'}:
        from astropy.table import Table

        return Table.read(catalog_path).to_pandas()
    if suffix in {'.csv', '.txt'}:
        return pd.read_csv(catalog_path)

    raise ValueError(f"Unsupported catalog format '{suffix}'. Use parquet, FITS, or CSV.")


def load_sample_catalog(name: str = 'raw_merger_flags', required: bool = True) -> pd.DataFrame | None:
    """Load one of the profile-managed Hyrax/TNG sample catalogs."""
    if name not in SAMPLE_CATALOG_PATHS:
        valid = ', '.join(sorted(SAMPLE_CATALOG_PATHS))
        raise KeyError(f"Unknown sample catalog '{name}'. Choose one of: {valid}")

    path = Path(SAMPLE_CATALOG_PATHS[name])
    if not path.exists():
        message = f"Sample catalog '{name}' is not available at {path}. Check path_profiles.toml or HYRAX_PROFILE."
        if required:
            raise FileNotFoundError(message)
        print(f'Skipping sample catalog: {message}')
        return None

    catalog = load_external_catalog(path)
    print(f"Loaded sample catalog '{name}': {len(catalog):,} rows from {path}")
    return catalog


def load_named_catalog(name: str, required: bool = True) -> pd.DataFrame | None:
    """Load an optional xmatch catalog by XCATALOG_PATHS key."""
    if name not in XCATALOG_PATHS:
        valid = ', '.join(sorted(XCATALOG_PATHS))
        raise KeyError(f"Unknown xmatch catalog '{name}'. Choose one of: {valid}")

    path = XCATALOG_PATHS[name]
    if not path.exists():
        message = (
            f"Catalog '{name}' is not available at {path}. "
            'Update path_profiles.toml, set HYRAX_PROFILE, or place a local copy there.'
        )
        if required:
            raise FileNotFoundError(message)
        print(f'Skipping optional xmatch catalog: {message}')
        return None

    catalog = load_external_catalog(path)
    print(f"Loaded xmatch catalog '{name}': {len(catalog):,} rows from {path}")
    return catalog


def _decode_scalar(value):
    """Decode byte strings from FITS tables while leaving other values unchanged."""
    if isinstance(value, (bytes, bytearray)):
        return value.decode('utf-8').strip()
    return value


def normalize_object_ids(values) -> pd.Series:
    """Normalize object IDs to comparable nullable strings."""
    ids = pd.Series(values, copy=False).map(_decode_scalar)
    missing = ids.isna()
    numeric = pd.to_numeric(ids, errors='coerce')

    if (~missing).any() and numeric.loc[~missing].notna().all():
        normalized = numeric.astype('Int64').astype('string')
    else:
        normalized = ids.astype('string').str.strip()

    return normalized.mask(missing)


def resolve_catalog_id_column(catalog: pd.DataFrame, catalog_id_column: str | None = None) -> str:
    """Find the catalog object-ID column used to match rows back to UMAP metadata."""
    candidates = [
        catalog_id_column,
        'object_id',
        'rubin_object_id',
        'objectId',
        'objectId_data',
        'id',
    ]

    for candidate in candidates:
        if candidate is not None and candidate in catalog.columns:
            return candidate

    raise KeyError(
        'Could not find an object ID column in the catalog. '
        'Pass catalog_id_column explicitly.'
    )


def first_present_column(catalog: pd.DataFrame, candidates: Iterable[str]) -> str | None:
    """Return the first candidate column present in a catalog."""
    for candidate in candidates:
        if candidate in catalog.columns:
            return candidate
    return None


def build_default_overlay_groups(catalog: pd.DataFrame) -> dict[str, list[dict]]:
    """Build useful overlay groups from the columns present in the loaded catalog."""
    groups: dict[str, list[dict]] = {}

    recent_specs = [
        ('has_major_past_1gyr', 'tab:red', 'x', 'Major past 1 Gyr'),
        ('has_minor_past_1gyr', 'tab:green', 's', 'Minor past 1 Gyr'),
        ('has_mini_past_1gyr', 'tab:blue', '.', 'Mini past 1 Gyr'),
    ]
    recent = [
        {'key': key, 'threshold': 0.5, 'color': color, 'marker': marker, 'label': label, 's': 10}
        for key, color, marker, label in recent_specs
        if key in catalog.columns
    ]
    if recent:
        groups['recent_merger_flags'] = recent

    future_specs = [
        ('has_major_future_1gyr', 'tab:red', 'x', 'Major future 1 Gyr'),
        ('has_minor_future_1gyr', 'tab:green', 's', 'Minor future 1 Gyr'),
        ('has_mini_future_1gyr', 'tab:blue', '.', 'Mini future 1 Gyr'),
    ]
    future = [
        {'key': key, 'threshold': 0.5, 'color': color, 'marker': marker, 'label': label, 's': 10}
        for key, color, marker, label in future_specs
        if key in catalog.columns
    ]
    if future:
        groups['future_merger_flags'] = future

    time_since = []
    for merger_type, color, marker in [
        ('Mini', 'tab:blue', '.'),
        ('Minor', 'tab:green', 's'),
        ('Major', 'tab:red', 'x'),
    ]:
        key = first_present_column(catalog, [
            f'{merger_type}_TimeSinceMerger',
            f'{merger_type.lower()}_time_since_merger',
        ])
        if key is not None:
            overlay = {
                'key': key,
                'min_value': 0.0,
                'include_min': True,
                'color': color,
                'marker': marker,
                'label': f'{merger_type} time since merger',
                's': 10,
            }
            if TIME_SINCE_MERGER_MAX_GYR is not None:
                overlay['max_value'] = float(TIME_SINCE_MERGER_MAX_GYR)
                overlay['include_max'] = True
                overlay['label'] = f'{merger_type} time since <= {float(TIME_SINCE_MERGER_MAX_GYR):g} Gyr'
            time_since.append(overlay)
    if time_since:
        groups['time_since_merger'] = time_since

    count_since = []
    for merger_type, color, marker in [
        ('Major', 'tab:red', 'x'),
        ('Minor', 'tab:green', 's'),
        ('Mini', 'tab:blue', '.'),
    ]:
        key = first_present_column(catalog, [
            f'{merger_type}_CountSince1Gyr',
            f'{merger_type.lower()}_count_since_1gyr',
        ])
        if key is not None:
            count_since.append({
                'key': key,
                'threshold': 1,
                'color': color,
                'marker': marker,
                'label': f'{merger_type} count since 1 Gyr >= 1',
                's': 10,
            })
    if count_since:
        groups['count_since_merger_1gyr'] = count_since

    if 'merging_merger_fraction' in catalog.columns:
        groups['euclid_vissyn_merger_fraction'] = [
            {'key': 'merging_merger_fraction', 'threshold': 0.7, 'color': 'red', 'marker': 'x', 'label': 'Euclid VisSyn mergers', 's': 10},
        ]

    return groups


def choose_overlay_group(overlay_groups: dict[str, list[dict]], preferred: str = 'recent_merger_flags') -> str:
    """Choose the preferred overlay group, or the first available group."""
    if preferred in overlay_groups:
        return preferred
    if overlay_groups:
        return next(iter(overlay_groups))
    raise ValueError(
        'No usable overlay columns were found in the loaded catalog. '
        'Expected columns like has_major_past_1gyr or Major_TimeSinceMerger.'
    )


def summarize_overlays(catalog: pd.DataFrame, overlays: list[dict]) -> pd.DataFrame:
    """Return catalog-row counts for each overlay selection."""
    rows = []
    for overlay in overlays:
        mask = _overlay_row_mask(catalog, overlay).fillna(False)
        key = overlay.get('key')
        row = {
            'label': overlay.get('label', overlay.get('key')),
            'key': key,
            'selected_catalog_rows': int(mask.sum()),
        }
        if key in catalog.columns:
            values = pd.to_numeric(catalog.loc[mask, key], errors='coerce')
            values = values[np.isfinite(values)]
            if len(values) > 0:
                row['selected_min'] = float(values.min())
                row['selected_median'] = float(values.median())
                row['selected_max'] = float(values.max())
        rows.append(row)
    return pd.DataFrame(rows)


def extract_umap_info(
    run: int,
    expt: int,
    base_directory: str | Path | None = None,
) -> tuple[Path, Path | None, Path]:
    """Resolve UMAP output, inference output, and config paths for one run/expt."""
    base_directory = Path(base_directory or HYRAX_RUN_BASE)

    try:
        umap_dir, inference_dir, config_file = sup.extract_umap_and_inference_info(
            run,
            expt,
            base_directory=base_directory,
        )
    except ValueError:
        umap_dir, config_file = sup.extract_umap_info(
            run,
            expt,
            base_directory=base_directory,
        )
        inference_dir = None

    return (
        Path(umap_dir),
        Path(inference_dir) if inference_dir is not None else None,
        Path(config_file),
    )


In [ ]:
def _extract_metadata_column(metadata_obj, field_name: str):
    """Extract one metadata field from dict, DataFrame, structured array, or array payloads."""
    if isinstance(metadata_obj, dict):
        if field_name in metadata_obj:
            return np.asarray(metadata_obj[field_name])
        if len(metadata_obj) == 1:
            return np.asarray(next(iter(metadata_obj.values())))
        return None

    if hasattr(metadata_obj, 'columns'):
        columns = list(metadata_obj.columns)
        if field_name in columns:
            return metadata_obj[field_name].to_numpy()
        if len(columns) == 1:
            return metadata_obj[columns[0]].to_numpy()
        return None

    dtype = getattr(metadata_obj, 'dtype', None)
    names = getattr(dtype, 'names', None)
    if names:
        if field_name in names:
            return np.asarray(metadata_obj[field_name])
        if len(names) == 1:
            return np.asarray(metadata_obj[names[0]])
        return None

    array = np.asarray(metadata_obj)
    if array.ndim == 1:
        return array
    if array.ndim == 2 and array.shape[1] == 1:
        return array[:, 0]
    return None


def get_umap_with_ids(
    config=None,
    input_dir: str | Path | None = None,
    suppress_logs: bool = True,
    id_field: str = 'objectId_data',
) -> dict:
    """Load UMAP coordinates and the object IDs used for catalog matching."""
    from hyrax.data_sets.inference_dataset import InferenceDataSet

    if suppress_logs:
        logging.disable(logging.CRITICAL)

    try:
        umap_results = InferenceDataSet(config, results_dir=input_dir, verb='umap')
        points = np.array([point.numpy() for point in umap_results])
        all_indices = list(range(len(umap_results)))
        available_fields = list(umap_results.metadata_fields())
    finally:
        if suppress_logs:
            logging.disable(logging.NOTSET)

    if points.ndim != 2 or points.shape[1] < 2:
        raise ValueError(f'UMAP results must be a 2D array with at least two columns; got {points.shape}')

    preferred_fields = [
        id_field,
        'objectId_data',
        'object_id_data',
        'objectId',
        'object_id',
        'rubin_object_id',
        'id',
    ]

    candidate_fields = []
    for field in preferred_fields:
        if field is not None and field not in candidate_fields:
            candidate_fields.append(field)

    candidate_fields = [field for field in candidate_fields if field in available_fields] + [
        field for field in candidate_fields if field not in available_fields
    ]

    attempts = []
    rubin_ids = None
    resolved_field = None

    for candidate in candidate_fields:
        try:
            metadata = umap_results.metadata(all_indices, [candidate])
            extracted = _extract_metadata_column(metadata, candidate)
            if extracted is None:
                attempts.append(f'{candidate}: not found in metadata payload')
                continue
            if len(extracted) != len(umap_results):
                attempts.append(f'{candidate}: length mismatch ({len(extracted)} vs {len(umap_results)})')
                continue

            rubin_ids = np.asarray(extracted)
            resolved_field = candidate
            break
        except Exception as exc:
            attempts.append(f'{candidate}: {exc}')

    if rubin_ids is None:
        raise KeyError(
            'Could not extract object IDs from UMAP metadata. '
            f'Available metadata fields: {available_fields}. '
            f"Attempts: {'; '.join(attempts)}"
        )

    return {
        'x': points[:, 0],
        'y': points[:, 1],
        'rubin_ids': rubin_ids,
        'id_field': resolved_field,
        'umap_results': umap_results,
    }


In [ ]:
def get_highdim_data(
    config=None,
    inference_dir: str | Path | None = None,
    umap_ids=None,
    suppress_logs: bool = True,
) -> np.ndarray | None:
    """Load high-dimensional latent vectors and align them to UMAP ID order."""
    if inference_dir is None:
        return None

    inference_dir = Path(inference_dir)
    result_ids = None

    try:
        points, result_ids = sup.load_result_tensors(inference_dir, flatten=True)
    except Exception as batch_error:
        from hyrax.data_sets.inference_dataset import InferenceDataSet

        if suppress_logs:
            logging.disable(logging.CRITICAL)
        try:
            inference_results = InferenceDataSet(config, results_dir=inference_dir)
            points = np.array([point.numpy() for point in inference_results])
        finally:
            if suppress_logs:
                logging.disable(logging.NOTSET)

        points = points.reshape(points.shape[0], -1)
        print(f'Loaded high-dimensional data through Hyrax fallback after batch read failed: {batch_error}')

    if umap_ids is None:
        return points

    if result_ids is None:
        if len(points) != len(umap_ids):
            raise ValueError(
                'High-dimensional results did not expose IDs and length does not match UMAP data: '
                f'{len(points)} vs {len(umap_ids)}'
            )
        return points

    highdim_lookup = pd.DataFrame({
        '_match_id': normalize_object_ids(result_ids),
        '_hd_index': np.arange(len(points)),
    }).dropna(subset=['_match_id'])

    if highdim_lookup['_match_id'].duplicated().any():
        duplicates = highdim_lookup.loc[highdim_lookup['_match_id'].duplicated(), '_match_id'].head().tolist()
        raise ValueError(f'High-dimensional result IDs contain duplicates; first duplicates: {duplicates}')

    umap_lookup = pd.DataFrame({
        '_match_id': normalize_object_ids(umap_ids),
        '_umap_index': np.arange(len(umap_ids)),
    })

    aligned = umap_lookup.merge(highdim_lookup, on='_match_id', how='left', validate='many_to_one')
    missing = aligned['_hd_index'].isna()
    if missing.any():
        preview = aligned.loc[missing, '_match_id'].head().tolist()
        raise ValueError(
            f'{int(missing.sum())} UMAP IDs are missing from the high-dimensional inference results. '
            f'First missing IDs: {preview}'
        )

    return points[aligned['_hd_index'].to_numpy(dtype=int)]


In [ ]:
from sklearn.neighbors import NearestNeighbors
try:
    from sklearn.cluster import HDBSCAN
except ImportError:
    from hdbscan import HDBSCAN
from scipy.spatial.distance import cdist


def mnln_ratio(coords, labeled_mask, n_permutations=500, seed=42):
    """Median Nearest Labeled Neighbor (MNLN) distance ratio with permutation test.

    For each labeled point, compute the distance to its nearest *other* labeled
    point.  Take the median.  Compare to a null distribution obtained by randomly
    picking the same number of points and computing their median nearest-same-set
    distance.

    Parameters
    ----------
    coords : np.ndarray, shape (N, D)
        Coordinates of all points (2-D UMAP or high-dim latent).
    labeled_mask : np.ndarray of bool, shape (N,)
        True for labeled points.
    n_permutations : int
        Number of random shuffles for the null distribution.
    seed : int
        Random seed.

    Returns
    -------
    dict with keys:
        'ratio'       : float – observed_median / expected_median  (< 1 = clustered)
        'observed'    : float – observed median NLN distance
        'expected'    : float – mean of null distribution medians
        'std_null'    : float – std of null distribution
        'z_score'     : float – (observed - expected) / std_null
        'p_value'     : float – fraction of null medians <= observed
        'n_labeled'   : int
    """
    rng = np.random.default_rng(seed)
    N = len(coords)
    n_labeled = int(labeled_mask.sum())

    if n_labeled < 2:
        return {'ratio': np.nan, 'observed': np.nan, 'expected': np.nan,
                'std_null': np.nan, 'z_score': np.nan, 'p_value': np.nan,
                'n_labeled': n_labeled}

    def _median_nln(indices):
        """Median nearest-neighbor distance within a set of point indices."""
        pts = coords[indices]
        if len(pts) < 2:
            return np.nan
        # Pairwise distances within the set
        dists = cdist(pts, pts)
        np.fill_diagonal(dists, np.inf)
        nn_dists = dists.min(axis=1)
        return np.median(nn_dists)

    labeled_indices = np.where(labeled_mask)[0]
    observed = _median_nln(labeled_indices)

    # Null distribution
    null_medians = np.empty(n_permutations)
    for p in range(n_permutations):
        rand_indices = rng.choice(N, size=n_labeled, replace=False)
        null_medians[p] = _median_nln(rand_indices)

    expected = null_medians.mean()
    std_null = null_medians.std()
    ratio = observed / expected if expected > 0 else np.nan
    z_score = (observed - expected) / std_null if std_null > 0 else np.nan
    # One-sided: how often is the null median <= observed (i.e., as compact or more)
    p_value = (np.sum(null_medians <= observed) + 1) / (n_permutations + 1)

    return {
        'ratio': ratio,
        'observed': observed,
        'expected': expected,
        'std_null': std_null,
        'z_score': z_score,
        'p_value': p_value,
        'n_labeled': n_labeled,
    }


def cmc_gini(coords, labeled_mask, min_cluster_size=15, n_permutations=500, seed=42):
    """Cluster Membership Concentration via Gini coefficient.

    Run HDBSCAN on all points to partition the space into regions.  Then measure
    how non-uniformly the labeled points distribute across those clusters using
    the Gini coefficient.  Compare to a null distribution from random label
    assignments.

    Parameters
    ----------
    coords : np.ndarray, shape (N, D)
        Coordinates of all points.
    labeled_mask : np.ndarray of bool, shape (N,)
        True for labeled points.
    min_cluster_size : int
        HDBSCAN min_cluster_size parameter.
    n_permutations : int
        Permutation count for null distribution.
    seed : int
        Random seed.

    Returns
    -------
    dict with keys:
        'gini'          : float – observed Gini coefficient (higher = more concentrated)
        'expected_gini' : float – mean Gini under random assignment
        'z_score'       : float
        'p_value'       : float – fraction of null Ginis >= observed
        'n_labeled'     : int
        'n_clusters'    : int   – number of HDBSCAN clusters found
        'cluster_counts': dict  – {cluster_label: count_of_labeled_points}
    """
    rng = np.random.default_rng(seed)
    N = len(coords)
    n_labeled = int(labeled_mask.sum())

    if n_labeled < 2:
        return {'gini': np.nan, 'expected_gini': np.nan, 'z_score': np.nan,
                'p_value': np.nan, 'n_labeled': n_labeled, 'n_clusters': 0,
                'cluster_counts': {}}

    # Cluster all points
    clusterer = HDBSCAN(min_cluster_size=min_cluster_size)
    cluster_labels = clusterer.fit_predict(coords)

    unique_clusters = np.unique(cluster_labels)
    # Include noise (-1) as its own bin — labeled points in noise are "elsewhere"
    n_clusters = len(unique_clusters)

    if n_clusters < 2:
        return {'gini': np.nan, 'expected_gini': np.nan, 'z_score': np.nan,
                'p_value': np.nan, 'n_labeled': n_labeled, 'n_clusters': n_clusters,
                'cluster_counts': {}}

    def _gini(mask):
        """Gini coefficient of labeled-point counts across clusters."""
        counts = np.array([np.sum(mask & (cluster_labels == c)) for c in unique_clusters],
                          dtype=float)
        total = counts.sum()
        if total == 0:
            return 0.0
        counts_sorted = np.sort(counts)
        n = len(counts_sorted)
        index = np.arange(1, n + 1)
        return (2 * np.sum(index * counts_sorted) - (n + 1) * total) / (n * total)

    observed_gini = _gini(labeled_mask)

    # Record cluster distribution
    cluster_counts = {}
    for c in unique_clusters:
        cnt = int(np.sum(labeled_mask & (cluster_labels == c)))
        if cnt > 0:
            cluster_counts[int(c)] = cnt

    # Null distribution
    null_ginis = np.empty(n_permutations)
    for p in range(n_permutations):
        perm_mask = np.zeros(N, dtype=bool)
        perm_mask[rng.choice(N, size=n_labeled, replace=False)] = True
        null_ginis[p] = _gini(perm_mask)

    expected_gini = null_ginis.mean()
    std_null = null_ginis.std()
    z_score = (observed_gini - expected_gini) / std_null if std_null > 0 else np.nan
    p_value = (np.sum(null_ginis >= observed_gini) + 1) / (n_permutations + 1)

    return {
        'gini': observed_gini,
        'expected_gini': expected_gini,
        'z_score': z_score,
        'p_value': p_value,
        'n_labeled': n_labeled,
        'n_clusters': int(len(unique_clusters[unique_clusters != -1])),
        'cluster_counts': cluster_counts,
    }


In [ ]:
def _overlay_row_mask(catalog: pd.DataFrame, overlay: dict) -> pd.Series:
    """Return the catalog-row mask requested by one overlay specification."""
    key = overlay.get('key')
    if key is None:
        return pd.Series(True, index=catalog.index)

    if key not in catalog.columns:
        raise KeyError(f"Catalog column '{key}' not found. Available columns include: {list(catalog.columns[:20])}")

    values = pd.to_numeric(catalog[key], errors='coerce')

    if 'min_value' in overlay or 'max_value' in overlay:
        mask = values.notna() & np.isfinite(values)
        if overlay.get('min_value') is not None:
            if overlay.get('include_min', True):
                mask &= values >= overlay['min_value']
            else:
                mask &= values > overlay['min_value']
        if overlay.get('max_value') is not None:
            if overlay.get('include_max', True):
                mask &= values <= overlay['max_value']
            else:
                mask &= values < overlay['max_value']
        return mask

    threshold = overlay.get('threshold', 0.0)
    comparator = overlay.get('comparator', '>=')
    if comparator == '>=':
        return values >= threshold
    if comparator == '>':
        return values > threshold
    if comparator == '<=':
        return values <= threshold
    if comparator == '<':
        return values < threshold
    if comparator == '==':
        return values == threshold
    raise ValueError(f"Unsupported comparator '{comparator}'")


def overlay_labeled_mask(
    umap_data: dict,
    catalog: pd.DataFrame,
    overlay: dict,
    catalog_id_column: str | None = None,
) -> np.ndarray:
    """Build a boolean mask over UMAP points for one overlay selection."""
    catalog_id_column = resolve_catalog_id_column(catalog, catalog_id_column)
    selected = catalog.loc[_overlay_row_mask(catalog, overlay).fillna(False), [catalog_id_column]].copy()

    if selected.empty:
        return np.zeros(len(umap_data['rubin_ids']), dtype=bool)

    selected_ids = set(normalize_object_ids(selected[catalog_id_column]).dropna().drop_duplicates())
    umap_ids = normalize_object_ids(umap_data['rubin_ids'])
    return umap_ids.isin(selected_ids).to_numpy(dtype=bool)


def compute_overlay_metrics(
    umap_data,
    catalog,
    overlays,
    n_permutations=500,
    min_cluster_size=15,
    highdim_coords=None,
    catalog_id_column: str | None = None,
):
    """Compute MNLN ratio and CMC-Gini for each overlay."""
    if catalog is None:
        raise ValueError('A catalog DataFrame is required to compute overlay metrics.')

    x = umap_data['x']
    y = umap_data['y']
    umap_coords = np.column_stack([x, y])
    n_points = len(umap_coords)

    if highdim_coords is not None and len(highdim_coords) != n_points:
        raise ValueError(
            'highdim_coords must be aligned to UMAP order and have the same length: '
            f'{len(highdim_coords)} vs {n_points}'
        )

    results = []
    for overlay in overlays:
        label = overlay.get('label', str(overlay.get('key')))
        labeled_mask = overlay_labeled_mask(
            umap_data,
            catalog,
            overlay,
            catalog_id_column=catalog_id_column,
        )
        n_matched = int(labeled_mask.sum())

        entry = {'label': label, 'n_matched': n_matched}
        entry['mnln_2d'] = mnln_ratio(
            umap_coords,
            labeled_mask,
            n_permutations=n_permutations,
        )
        entry['cmc_2d'] = cmc_gini(
            umap_coords,
            labeled_mask,
            min_cluster_size=min_cluster_size,
            n_permutations=n_permutations,
        )

        if highdim_coords is not None:
            entry['mnln_hd'] = mnln_ratio(
                highdim_coords,
                labeled_mask,
                n_permutations=n_permutations,
            )
            entry['cmc_hd'] = cmc_gini(
                highdim_coords,
                labeled_mask,
                min_cluster_size=min_cluster_size,
                n_permutations=n_permutations,
            )

        results.append(entry)

    return results


In [ ]:
def format_metric_text(metric_results, include_hd=False):
    """Format metric results into annotation strings for plot subtitles."""
    lines = []
    for m in metric_results:
        mnln = m['mnln_2d']
        cmc = m['cmc_2d']

        # Format MNLN p-value
        if np.isnan(mnln['p_value']):
            mp = "n/a"
        elif mnln['p_value'] < 0.001:
            mp = "p<.001"
        elif mnln['p_value'] < 0.01:
            mp = f"p={mnln['p_value']:.3f}"
        else:
            mp = f"p={mnln['p_value']:.2f}"

        # Format CMC p-value
        if np.isnan(cmc['p_value']):
            cp = "n/a"
        elif cmc['p_value'] < 0.001:
            cp = "p<.001"
        elif cmc['p_value'] < 0.01:
            cp = f"p={cmc['p_value']:.3f}"
        else:
            cp = f"p={cmc['p_value']:.2f}"

        line = (f"{m['label']} (n={m['n_matched']}): "
                f"MNLN={mnln['ratio']:.2f} ({mp}), "
                f"Gini={cmc['gini']:.2f} ({cp})")

        if include_hd and 'mnln_hd' in m:
            mnln_hd = m['mnln_hd']
            cmc_hd = m['cmc_hd']
            if np.isnan(mnln_hd['p_value']):
                mp_hd = "n/a"
            elif mnln_hd['p_value'] < 0.001:
                mp_hd = "p<.001"
            else:
                mp_hd = f"p={mnln_hd['p_value']:.2f}"
            if np.isnan(cmc_hd['p_value']):
                cp_hd = "n/a"
            elif cmc_hd['p_value'] < 0.001:
                cp_hd = "p<.001"
            else:
                cp_hd = f"p={cmc_hd['p_value']:.2f}"
            line += (f"\n  HD: MNLN={mnln_hd['ratio']:.2f} ({mp_hd}), "
                     f"Gini={cmc_hd['gini']:.2f} ({cp_hd})")

        lines.append(line)
    return "\n".join(lines)

In [ ]:
def matched_overlay_points(
    umap_data: dict,
    catalog: pd.DataFrame,
    overlay: dict,
    catalog_id_column: str | None = None,
) -> pd.DataFrame:
    """Return UMAP rows that match one catalog overlay."""
    catalog_id_column = resolve_catalog_id_column(catalog, catalog_id_column)
    key = overlay.get('key')
    selected_columns = [catalog_id_column]
    if key is not None and key in catalog.columns:
        selected_columns.append(key)

    selected = catalog.loc[_overlay_row_mask(catalog, overlay).fillna(False), selected_columns].copy()
    if selected.empty:
        return pd.DataFrame(columns=['x', 'y', '_match_id'])

    selected['_match_id'] = normalize_object_ids(selected[catalog_id_column])
    selected = selected.dropna(subset=['_match_id']).drop_duplicates('_match_id')

    umap_lookup = pd.DataFrame({
        'x': umap_data['x'],
        'y': umap_data['y'],
        '_match_id': normalize_object_ids(umap_data['rubin_ids']),
    }).dropna(subset=['_match_id'])

    return selected.merge(umap_lookup, on='_match_id', how='inner')


def plot_umap_with_multi_overlay(
    ax,
    umap_data,
    catalog,
    overlays,
    catalog_id_column: str | None = None,
    alpha_background=0.1,
    s_background=1,
    title=None,
    show_legend=True,
    density=False,
    log_colorbar=False,
    density_cmap='viridis',
):
    """Plot UMAP with multiple catalog overlays."""
    if catalog is None:
        raise ValueError('A catalog DataFrame is required for overlay plotting.')

    x = umap_data['x']
    y = umap_data['y']

    if density:
        norm = LogNorm() if log_colorbar else None
        hb = ax.hexbin(x, y, gridsize=50, cmap=density_cmap, norm=norm)
        plt.colorbar(hb, ax=ax, label='Count')
    else:
        ax.scatter(x, y, alpha=alpha_background, s=s_background, c='gray', label='All', linewidths=0)

    for overlay in overlays:
        color = overlay['color']
        marker = overlay['marker']
        label = overlay['label']
        s = overlay.get('s', 20)
        alpha = overlay.get('alpha', 1.0)
        edgecolors = overlay.get('edgecolors', 'none')
        linewidths = overlay.get('linewidths', 1.0)

        matched = matched_overlay_points(
            umap_data,
            catalog,
            overlay,
            catalog_id_column=catalog_id_column,
        )
        if matched.empty:
            continue

        ax.scatter(
            matched['x'].to_numpy(),
            matched['y'].to_numpy(),
            alpha=alpha,
            s=s,
            c=color,
            marker=marker,
            label=f'{label} (n={len(matched)})',
            edgecolors=edgecolors,
            linewidths=linewidths,
        )

    if show_legend:
        ax.legend(loc='best', fontsize='small')
    if title:
        ax.set_title(title)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    return ax


In [ ]:
def plot_multiple_umaps_with_metrics(
    runs,
    expts,
    catalog,
    overlays,
    catalog_id_column: str | None = None,
    n_permutations=500,
    min_cluster_size=15,
    include_highdim=False,
    base_directory: str | Path | None = None,
    ncols=2,
    figsize=None,
    dpi=150,
    save_path=None,
    suptitle=None,
    suppress_logs=True,
    alpha_background=0.5,
    s_background=1,
    show_legend=True,
    density=False,
    log_colorbar=False,
    density_cmap='viridis',
):
    """Multi-panel UMAP plots with overlay markers and metric annotations."""
    import math

    try:
        from tqdm.notebook import tqdm
    except Exception:
        tqdm = lambda iterable, total=None, desc=None: iterable

    if catalog is None:
        print('No catalog loaded; skipping this plot.')
        return None, []

    if len(runs) != len(expts):
        raise ValueError('runs and expts arrays must be the same length')

    n_plots = len(runs)
    nrows = math.ceil(n_plots / ncols)
    if figsize is None:
        figsize = (ncols * 5, nrows * 4.5)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, dpi=dpi, squeeze=False)
    axes = axes.ravel()
    all_metrics = []

    for i, (run, expt) in enumerate(tqdm(list(zip(runs, expts)), total=n_plots, desc='Panels')):
        ax = axes[i]

        try:
            umap_dir, inference_dir, config_file = extract_umap_info(
                run,
                expt,
                base_directory=base_directory,
            )

            if suppress_logs:
                logging.disable(logging.CRITICAL)
            try:
                h = hyrax.Hyrax(config_file=config_file)
            finally:
                if suppress_logs:
                    logging.disable(logging.NOTSET)

            umap_data = get_umap_with_ids(
                config=h.config,
                input_dir=umap_dir,
                suppress_logs=suppress_logs,
            )

            hd = None
            if include_highdim:
                if inference_dir is None:
                    print(f'Run {run}, Expt {expt}: no inference_dir in config; skipping high-dimensional metrics.')
                else:
                    try:
                        hd = get_highdim_data(
                            config=h.config,
                            inference_dir=inference_dir,
                            umap_ids=umap_data['rubin_ids'],
                            suppress_logs=suppress_logs,
                        )
                    except Exception as exc:
                        print(f'Run {run}, Expt {expt}: could not load aligned high-dimensional data; skipping HD metrics. {exc}')

            metrics = compute_overlay_metrics(
                umap_data,
                catalog,
                overlays,
                n_permutations=n_permutations,
                min_cluster_size=min_cluster_size,
                highdim_coords=hd,
                catalog_id_column=catalog_id_column,
            )
            all_metrics.append((run, expt, metrics))

            title = f'Run {run}, Expt {expt}'
            plot_umap_with_multi_overlay(
                ax,
                umap_data,
                catalog,
                overlays,
                catalog_id_column=catalog_id_column,
                alpha_background=alpha_background,
                s_background=s_background,
                title=title,
                show_legend=show_legend,
                density=density,
                log_colorbar=log_colorbar,
                density_cmap=density_cmap,
            )

            annotation = format_metric_text(metrics, include_hd=include_highdim and hd is not None)
            ax.text(
                0.02,
                0.02,
                annotation,
                transform=ax.transAxes,
                fontsize=6,
                verticalalignment='bottom',
                fontfamily='monospace',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
            )

        except Exception as exc:
            import traceback

            ax.text(
                0.5,
                0.5,
                f'Error\nRun {run}, Expt {expt}\n{exc}',
                ha='center',
                va='center',
                transform=ax.transAxes,
                fontsize='small',
            )
            ax.set_title(f'Run {run}, Expt {expt} - ERROR')
            traceback.print_exc()

    for j in range(n_plots, len(axes)):
        axes[j].axis('off')

    if suptitle:
        fig.suptitle(suptitle, fontsize=14, y=1.01)

    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    else:
        plt.show()

    return fig, all_metrics


In [ ]:
def print_metrics_summary(all_metrics, include_hd=False):
    """Print a summary table of metrics across runs/experiments."""
    print(f"{'Run':>4} {'Expt':>5}  {'Overlay':<25} {'n':>4}  "
          f"{'MNLN':>6} {'p_mnln':>7} {'Gini':>5} {'p_gini':>7} {'nClus':>5}", end="")
    if include_hd:
        print(f"  {'MNLN_HD':>7} {'p_HD':>7} {'Gini_HD':>7} {'p_gHD':>7}", end="")
    print()
    print("-" * (75 + (35 if include_hd else 0)))

    for run, expt, metrics in all_metrics:
        for m in metrics:
            mnln = m['mnln_2d']
            cmc = m['cmc_2d']
            p_mnln = f"{mnln['p_value']:.4f}" if not np.isnan(mnln['p_value']) else "n/a"
            p_gini = f"{cmc['p_value']:.4f}" if not np.isnan(cmc['p_value']) else "n/a"
            gini_str = f"{cmc['gini']:.3f}" if not np.isnan(cmc['gini']) else "n/a"
            print(f"{run:>4} {expt:>5}  {m['label']:<25} {m['n_matched']:>4}  "
                  f"{mnln['ratio']:>6.3f} {p_mnln:>7} {gini_str:>5} {p_gini:>7} "
                  f"{cmc['n_clusters']:>5}", end="")
            if include_hd and 'mnln_hd' in m:
                mnln_hd = m['mnln_hd']
                cmc_hd = m['cmc_hd']
                p_hd = f"{mnln_hd['p_value']:.4f}" if not np.isnan(mnln_hd['p_value']) else "n/a"
                g_hd = f"{cmc_hd['gini']:.3f}" if not np.isnan(cmc_hd['gini']) else "n/a"
                pg_hd = f"{cmc_hd['p_value']:.4f}" if not np.isnan(cmc_hd['p_value']) else "n/a"
                print(f"  {mnln_hd['ratio']:>7.3f} {p_hd:>7} {g_hd:>7} {pg_hd:>7}", end="")
            print()

## Load `catalog2.fits`


In [ ]:
print(f'Path profile: {paths.profile}')
print(f'Hyrax run base: {HYRAX_RUN_BASE}')
display(catalog_path_status())

# Always use catalog2.fits: this profile key resolves to paths.catalog('raw_merger_flags').
CATALOG_KEY = 'raw_merger_flags'
catalog = load_sample_catalog(CATALOG_KEY)
catalog_id_column = resolve_catalog_id_column(catalog)
overlay_groups = build_default_overlay_groups(catalog)
ACTIVE_OVERLAY_GROUP = choose_overlay_group(overlay_groups, preferred='time_since_merger')
overlays = overlay_groups[ACTIVE_OVERLAY_GROUP]

print(f'Active catalog: {CATALOG_KEY}')
print(f'Catalog path: {SAMPLE_CATALOG_PATHS[CATALOG_KEY]}')
print(f'Catalog ID column: {catalog_id_column}')
print(f'Available overlay groups: {list(overlay_groups)}')
print(f'Active overlay group: {ACTIVE_OVERLAY_GROUP}')
display(summarize_overlays(catalog, overlays))

preview_columns = [catalog_id_column]
preview_columns += [overlay['key'] for overlay in overlays if overlay.get('key') in catalog.columns]
preview_columns = list(dict.fromkeys(preview_columns))
display(catalog[preview_columns].head())

# Optional external catalogs are still available if you place them at the configured paths.
xmatch_catalogs = {
    name: load_named_catalog(name, required=False)
    for name in XCATALOG_PATHS
}


## Native Catalog Time-Since-Merger Overlays with Metrics

### Run 10


In [ ]:
expts = [7, 10, 12, 13, 14, 15, 18]
runs = [10] * len(expts)

fig, all_metrics = plot_multiple_umaps_with_metrics(
    runs,
    expts,
    catalog,
    overlays,
    catalog_id_column=catalog_id_column,
    n_permutations=DEFAULT_N_PERMUTATIONS,
    include_highdim=INCLUDE_HIGHDIM,
    ncols=2,
    alpha_background=0.5,
    show_legend=True,
    suptitle=f'Run 10 - {ACTIVE_OVERLAY_GROUP} from {CATALOG_KEY} catalog',
)


In [ ]:
if all_metrics:
    print_metrics_summary(all_metrics, include_hd=INCLUDE_HIGHDIM)
else:
    print('No metrics available for Run 10.')


### Run 11


In [ ]:
expts = [7, 10, 12, 13, 14]
runs = [11] * len(expts)

fig, all_metrics_r11 = plot_multiple_umaps_with_metrics(
    runs,
    expts,
    catalog,
    overlays,
    catalog_id_column=catalog_id_column,
    n_permutations=DEFAULT_N_PERMUTATIONS,
    include_highdim=INCLUDE_HIGHDIM,
    ncols=2,
    alpha_background=0.5,
    show_legend=True,
    suptitle=f'Run 11 - {ACTIVE_OVERLAY_GROUP} from {CATALOG_KEY} catalog',
)


In [ ]:
if all_metrics_r11:
    print_metrics_summary(all_metrics_r11, include_hd=INCLUDE_HIGHDIM)
else:
    print('No metrics available for Run 11.')


## Raw Time-Since-Merger Values


In [ ]:
time_since_overlays = overlay_groups.get('time_since_merger')
if time_since_overlays is None:
    print('No time-since-merger columns found in this catalog.')
    time_since_metrics = []
else:
    display(summarize_overlays(catalog, time_since_overlays))
    expts = [7, 10, 12, 13, 14, 15, 18]
    runs = [10] * len(expts)

    fig, time_since_metrics = plot_multiple_umaps_with_metrics(
        runs,
        expts,
        catalog,
        time_since_overlays,
        catalog_id_column=catalog_id_column,
        n_permutations=DEFAULT_N_PERMUTATIONS,
        include_highdim=INCLUDE_HIGHDIM,
        ncols=2,
        alpha_background=0.5,
        show_legend=True,
        suptitle=f'Run 10 - time-since-merger from {CATALOG_KEY} catalog',
    )


In [ ]:
if time_since_metrics:
    print_metrics_summary(time_since_metrics, include_hd=INCLUDE_HIGHDIM)
else:
    print('No time-since-merger metrics available.')


## Future Merger Flags


In [ ]:
future_overlays = overlay_groups.get('future_merger_flags')
if future_overlays is None:
    print('No future-merger flag columns found in this catalog.')
    future_metrics = []
else:
    display(summarize_overlays(catalog, future_overlays))
    expts = [7, 10, 12, 13, 14, 15, 18]
    runs = [10] * len(expts)

    fig, future_metrics = plot_multiple_umaps_with_metrics(
        runs,
        expts,
        catalog,
        future_overlays,
        catalog_id_column=catalog_id_column,
        n_permutations=DEFAULT_N_PERMUTATIONS,
        include_highdim=INCLUDE_HIGHDIM,
        ncols=2,
        alpha_background=0.5,
        show_legend=True,
        suptitle=f'Run 10 - future merger flags from {CATALOG_KEY} catalog',
    )


In [ ]:
if future_metrics:
    print_metrics_summary(future_metrics, include_hd=INCLUDE_HIGHDIM)
else:
    print('No future-merger metrics available.')


## Single-Run Deep Dive

For a single run/experiment, inspect the full metric output for the active native-catalog overlays.


In [ ]:
run, expt = 10, 12

try:
    umap_dir, inference_dir, config_file = extract_umap_info(run, expt)
    h = hyrax.Hyrax(config_file=config_file)
    umap_data = get_umap_with_ids(config=h.config, input_dir=umap_dir)
    hd = None
    if INCLUDE_HIGHDIM and inference_dir is not None:
        hd = get_highdim_data(h.config, inference_dir, umap_ids=umap_data['rubin_ids'])

    metrics = compute_overlay_metrics(
        umap_data,
        catalog,
        overlays,
        catalog_id_column=catalog_id_column,
        n_permutations=DEFAULT_N_PERMUTATIONS,
        highdim_coords=hd,
    )

    for m in metrics:
        print(f"\n=== {m['label']} (n={m['n_matched']}) ===")
        mnln = m['mnln_2d']
        cmc = m['cmc_2d']
        print('  2D UMAP:')
        print(f"    MNLN ratio:  {mnln['ratio']:.3f}  (observed={mnln['observed']:.3f}, expected={mnln['expected']:.3f}, z={mnln['z_score']:.2f}, p={mnln['p_value']:.4f})")
        print(f"    CMC Gini:    {cmc['gini']:.3f}  (expected={cmc['expected_gini']:.3f}, z={cmc['z_score']:.2f}, p={cmc['p_value']:.4f}, clusters={cmc['n_clusters']})")
        if cmc['cluster_counts']:
            print(f"    Cluster distribution: {cmc['cluster_counts']}")
        if 'mnln_hd' in m:
            mnln_hd = m['mnln_hd']
            cmc_hd = m['cmc_hd']
            print('  High-Dim Latent:')
            print(f"    MNLN ratio:  {mnln_hd['ratio']:.3f}  (observed={mnln_hd['observed']:.3f}, expected={mnln_hd['expected']:.3f}, z={mnln_hd['z_score']:.2f}, p={mnln_hd['p_value']:.4f})")
            print(f"    CMC Gini:    {cmc_hd['gini']:.3f}  (expected={cmc_hd['expected_gini']:.3f}, z={cmc_hd['z_score']:.2f}, p={cmc_hd['p_value']:.4f}, clusters={cmc_hd['n_clusters']})")
            if cmc_hd['cluster_counts']:
                print(f"    Cluster distribution: {cmc_hd['cluster_counts']}")
except Exception as exc:
    print(f'Deep dive failed for Run {run}, Expt {expt}: {exc}')
